# LeetCode 768 — Max Chunks To Make Sorted II bằng Python

- Link: https://leetcode.com/problems/max-chunks-to-make-sorted-ii/
- Độ khó: **Hard**
- Chủ đề: Mảng, Sắp xếp, Tiền tố max / hậu tố min, Ngăn xếp đơn điệu (Monotonic Stack), Đếm (Counter), Tham lam

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Ta muốn cắt mảng thành **nhiều mảnh nhất** sao cho: sắp xếp riêng từng mảnh rồi ghép lại theo thứ tự thì được **đúng mảng đã sắp xếp**. Phiên bản I (*769*) chỉ có hoán vị `0..n-1`; phiên bản II này cho phép **giá trị lớn và trùng nhau**, nên các mẹo "so với chỉ số" không còn dùng được.

Sau notebook này bạn sẽ:

1. Hiểu điều kiện cắt: "mọi phần tử bên trái ≤ mọi phần tử bên phải".
2. Viết lời giải vét cạn mọi cách cắt (baseline) và lời giải so khớp với mảng đã sắp bằng `Counter`.
3. Dùng **tiền tố max / hậu tố min** để kiểm tra mỗi vị trí cắt trong `O(1)`.
4. Dùng **tổng tiền tố** — một mẹo bất ngờ mà đúng.
5. Thành thạo **ngăn xếp đơn điệu** gộp mảnh — lời giải một lượt, rất hay gặp trong phỏng vấn.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `arr`. Chia `arr` thành các **đoạn liên tiếp** ("mảnh", chunk). Sắp xếp **từng mảnh** riêng rồi nối lại theo thứ tự cũ. Yêu cầu kết quả bằng `sorted(arr)`. Trả về **số mảnh lớn nhất** có thể.

**Ví dụ 1**

```text
Input: arr = [5,4,3,2,1]
Output: 1
Giải thích: chia thành 2 mảnh trở lên đều sai, ví dụ [5,4], [3,2,1] -> [4,5,1,2,3] không phải mảng tăng.
```

**Ví dụ 2**

```text
Input: arr = [2,1,3,4,4]
Output: 4
Giải thích: [2,1], [3], [4], [4] -> 4 mảnh. Không thể có 5 mảnh.
```

**Ràng buộc**

- `1 <= arr.length <= 2000`
- `0 <= arr[i] <= 10^8`

In [1]:
from typing import List

EXAMPLES = [([5, 4, 3, 2, 1], 1), ([2, 1, 3, 4, 4], 4)]

def apply_chunks(arr, cuts):
    """cuts: danh sách chỉ số i nghĩa là cắt SAU arr[i]. Trả về mảng sau khi sắp từng mảnh."""
    res, start = [], 0
    for c in list(cuts) + [len(arr) - 1]:
        res += sorted(arr[start:c + 1])
        start = c + 1
    return res

print(apply_chunks([2, 1, 3, 4, 4], [1, 2, 3]), sorted([2, 1, 3, 4, 4]))
print(apply_chunks([5, 4, 3, 2, 1], [1]))

[1, 2, 3, 4, 4] [1, 2, 3, 4, 4]
[4, 5, 1, 2, 3]


## 3. Kiến thức nền

### 3.1. Tiền tố max và hậu tố min

`left_max[i] = max(arr[0..i])`, `right_min[i] = min(arr[i..n-1])`. Tính trong một lượt mỗi mảng. `itertools.accumulate(arr, max)` cho tiền tố max gọn gàng.

In [2]:
from itertools import accumulate
arr = [2, 1, 3, 4, 4]
left_max = list(accumulate(arr, max))
right_min = list(accumulate(reversed(arr), min))[::-1]
print("left_max :", left_max)
print("right_min:", right_min)

left_max : [2, 2, 3, 4, 4]
right_min: [1, 1, 3, 4, 4]


### 3.2. `collections.Counter` để so sánh đa tập (multiset)

Hai đoạn có cùng các phần tử (kể cả số lần lặp) ⇔ `Counter` bằng nhau. Ta có thể duy trì **hiệu** của hai bộ đếm và theo dõi số khoá khác 0.

In [3]:
from collections import Counter
print(Counter([2, 1, 4]) == Counter([1, 4, 2]), Counter([4, 4]) == Counter([4]))

True False


### 3.3. Ngăn xếp đơn điệu (monotonic stack)

Ngăn xếp mà các phần tử từ đáy lên đỉnh **không giảm**. Khi đẩy phần tử mới nhỏ hơn đỉnh, ta pop các phần tử lớn hơn nó. Mẫu cơ bản: tìm "phần tử nhỏ hơn gần nhất bên trái".

In [4]:
def prev_smaller(a):
    st, res = [], []
    for x in a:
        while st and st[-1] >= x:
            st.pop()
        res.append(st[-1] if st else None)
        st.append(x)
    return res

print(prev_smaller([3, 1, 4, 1, 5, 9, 2, 6]))

[None, None, 1, None, 1, 5, 1, 2]


### 3.4. Sắp xếp ổn định và giá trị trùng

`sorted` trong Python ổn định; với giá trị trùng, thứ tự tương đối giữ nguyên. Ta sẽ dùng mẹo **gắn chỉ số** `(value, index)` để biến mảng có trùng thành hoán vị, đưa bài II về bài I.

In [5]:
arr = [2, 1, 3, 4, 4]
order = sorted(range(len(arr)), key=lambda i: (arr[i], i))
rank = [0] * len(arr)
for r, i in enumerate(order):
    rank[i] = r
print("rank =", rank)   # hoán vị 0..n-1 giữ thứ tự tương đối

rank = [1, 0, 2, 3, 4]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Khi nào được cắt sau vị trí `i`?

Sau khi sắp từng mảnh, phần bên trái chỗ cắt phải là **`i+1` phần tử nhỏ nhất** của mảng. Điều đó tương đương:

$$\max(arr[0..i]) \le \min(arr[i+1..n-1]).$$

Và quan trọng: **các vị trí cắt độc lập với nhau** — cắt được ở `i` hay không chỉ phụ thuộc vào `i`, không phụ thuộc các vết cắt khác. Vậy đáp án = **số vị trí cắt hợp lệ + 1**.

### 4.2. Tính tay ví dụ 2: `[2, 1, 3, 4, 4]`

| i | left_max[i] | right_min[i+1] | cắt được? |
|---|---|---|---|
| 0 | 2 | 1 | không |
| 1 | 2 | 3 | có |
| 2 | 3 | 4 | có |
| 3 | 4 | 4 | có (≤, giá trị bằng nhau vẫn cắt được) |

3 vết cắt → 4 mảnh. ✔

### 4.3. Tính tay ví dụ 1: `[5, 4, 3, 2, 1]`

`left_max` = 5,5,5,5; `right_min[i+1]` = 1,1,1,1 → không cắt được ở đâu → 1 mảnh.

In [6]:
arr = [2, 1, 3, 4, 4]
lm = list(accumulate(arr, max))
rm = list(accumulate(reversed(arr), min))[::-1]
print([(i, lm[i], rm[i + 1], lm[i] <= rm[i + 1]) for i in range(len(arr) - 1)])

[(0, 2, 1, False), (1, 2, 3, True), (2, 3, 4, True), (3, 4, 4, True)]


## 5. Cách 1 — Brute Force: thử mọi tập vết cắt

**Ý tưởng.** Có `n − 1` vị trí có thể cắt → `2^(n−1)` cách. Với mỗi cách, áp dụng và so với `sorted(arr)`; lấy số mảnh lớn nhất trong các cách hợp lệ.

In [7]:
from itertools import combinations

def brute(arr):
    n = len(arr)
    target = sorted(arr)
    best = 1
    for mask in range(1 << (n - 1)):
        cuts = [i for i in range(n - 1) if mask >> i & 1]
        if len(cuts) + 1 > best and apply_chunks(arr, cuts) == target:
            best = len(cuts) + 1
    return best

for arr, exp in EXAMPLES:
    print(arr, "->", brute(arr), "(kỳ vọng", exp, ")")

[5, 4, 3, 2, 1] -> 1 (kỳ vọng 1 )
[2, 1, 3, 4, 4] -> 4 (kỳ vọng 4 )


**Độ phức tạp.** `O(2^n · n log n)`. Với `n = 2000` là không tưởng; chỉ dùng cho `n ≤ 12` để kiểm thử.

**Nút thắt.** Ta chưa khai thác tính **độc lập** của các vết cắt (mục 4.1). Khi đã biết mỗi vết cắt kiểm tra riêng được, chỉ cần đếm.

## 6. Bước đột phá

**Insight 1 — Vết cắt độc lập.** Cắt sau `i` hợp lệ ⇔ `arr[0..i]` là đa tập của `i+1` phần tử nhỏ nhất ⇔ `max(arr[0..i]) <= min(arr[i+1..])`. Các điều kiện này không phụ thuộc nhau, nên đáp án = `1 + #{i : điều kiện đúng}`.

**Insight 2 — Nhiều cách kiểm tra điều kiện trong `O(1)` khấu hao:**

- So khớp đa tập với `sorted(arr)` bằng bộ đếm hiệu (Cách 2).
- So tổng tiền tố với tổng tiền tố của mảng đã sắp (Cách 3).
- Tiền tố max và hậu tố min (Cách 4).
- Ngăn xếp đơn điệu: mỗi mảnh đại diện bởi max của nó (Cách 5).

**Bất biến cho ngăn xếp (Cách 5).** Sau khi xử lý `arr[0..i]`, ngăn xếp chứa **max của từng mảnh** trong cách chia tối ưu của tiền tố, và dãy này **không giảm** từ đáy lên đỉnh.

**Quy tắc nếu… thì…** (ngăn xếp)

- Nếu `x >= đỉnh` → `x` mở một mảnh mới: đẩy `x`.
- Nếu `x < đỉnh` → `x` phải thuộc cùng mảnh với mọi mảnh có max `> x`: pop chúng, giữ lại max lớn nhất (đỉnh cũ), đẩy lại.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — So khớp với mảng đã sắp bằng bộ đếm hiệu

**Giải thích.** `s = sorted(arr)`. Duyệt `i`: `diff[arr[i]] += 1`, `diff[s[i]] -= 1`. Duy trì `nonzero` = số khoá có `diff ≠ 0`. Khi `nonzero == 0`, đa tập `arr[0..i]` trùng `s[0..i]` → cắt được.

In [8]:
def by_counter(arr):
    s = sorted(arr)
    diff = {}
    nonzero = 0
    chunks = 0
    for a, b in zip(arr, s):
        for key, d in ((a, 1), (b, -1)):
            before = diff.get(key, 0)
            after = before + d
            diff[key] = after
            if before == 0 and after != 0:
                nonzero += 1
            elif before != 0 and after == 0:
                nonzero -= 1
        if nonzero == 0:
            chunks += 1
    return chunks

for arr, exp in EXAMPLES:
    print(by_counter(arr), exp)

1 1
4 4


**Code nộp LeetCode (Cách 2).**

In [9]:
class SolutionCounter:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        from collections import defaultdict
        diff = defaultdict(int)
        nonzero = chunks = 0
        for a, b in zip(arr, sorted(arr)):
            diff[a] += 1
            if diff[a] == 0:
                nonzero -= 1
            elif diff[a] == 1:
                nonzero += 1
            diff[b] -= 1
            if diff[b] == 0:
                nonzero -= 1
            elif diff[b] == -1:
                nonzero += 1
            if nonzero == 0:
                chunks += 1
        return chunks

print([SolutionCounter().maxChunksToSorted(a[:]) for a, _ in EXAMPLES])

[1, 4]


Chú ý: khi `a == b`, `diff[a]` tăng lên 1 rồi giảm về 0 — `nonzero` tăng rồi giảm, kết quả đúng.

**Độ phức tạp.** Sắp `O(n log n)`, duyệt `O(n)`. Bộ nhớ `O(n)`.

### 7.2. Cách 3 — Tổng tiền tố (mẹo bất ngờ)

**Giải thích.** `sum(arr[0..i]) >= sum(s[0..i])` luôn đúng (`s[0..i]` là `i+1` phần tử nhỏ nhất, tổng nhỏ nhất). Dấu bằng xảy ra **khi và chỉ khi** `arr[0..i]` cũng là một tập `i+1` phần tử nhỏ nhất (nếu có phần tử bên trái lớn hơn một phần tử bên phải, hoán đổi chúng giảm tổng — mâu thuẫn với tính nhỏ nhất). Vậy cắt được ⇔ hai tổng tiền tố bằng nhau.

Python có số nguyên lớn tuỳ ý nên không lo tràn số (ở C++/Java dùng `long`).

In [10]:
def by_prefix_sum(arr):
    chunks = 0
    sa = ss = 0
    for a, b in zip(arr, sorted(arr)):
        sa += a
        ss += b
        if sa == ss:
            chunks += 1
    return chunks

for arr, exp in EXAMPLES:
    print(by_prefix_sum(arr), exp)

1 1
4 4


**Code nộp LeetCode (Cách 3).**

In [11]:
class SolutionPrefixSum:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        chunks = sa = ss = 0
        for a, b in zip(arr, sorted(arr)):
            sa += a
            ss += b
            chunks += sa == ss
        return chunks

print([SolutionPrefixSum().maxChunksToSorted(a[:]) for a, _ in EXAMPLES])

[1, 4]


**Độ phức tạp.** `O(n log n)`, bộ nhớ `O(n)` cho mảng sắp.

### 7.3. Cách 4 — Tiền tố max / hậu tố min

**Giải thích.** Tính `right_min[i] = min(arr[i..])` từ phải sang. Duyệt từ trái giữ `cur_max`; tại `i < n − 1`, nếu `cur_max <= right_min[i+1]` thì đếm một vết cắt. Đáp án = số vết cắt + 1.

In [12]:
def by_max_min(arr):
    n = len(arr)
    right_min = [0] * (n + 1)
    right_min[n] = float('inf')
    for i in range(n - 1, -1, -1):
        right_min[i] = min(arr[i], right_min[i + 1])
    chunks = 0
    cur_max = float('-inf')
    for i in range(n):
        cur_max = max(cur_max, arr[i])
        if cur_max <= right_min[i + 1]:
            chunks += 1
    return chunks

for arr, exp in EXAMPLES:
    print(by_max_min(arr), exp)

1 1
4 4


Mẹo: đặt `right_min[n] = +∞` → vị trí cuối luôn "cắt được" và tự đếm mảnh cuối, khỏi `+1`.

**Code nộp LeetCode (Cách 4).**

In [13]:
class SolutionMaxMin:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        n = len(arr)
        right_min = [float('inf')] * (n + 1)
        for i in range(n - 1, -1, -1):
            right_min[i] = min(arr[i], right_min[i + 1])
        chunks, cur_max = 0, float('-inf')
        for i in range(n):
            cur_max = max(cur_max, arr[i])
            if cur_max <= right_min[i + 1]:
                chunks += 1
        return chunks

print([SolutionMaxMin().maxChunksToSorted(a[:]) for a, _ in EXAMPLES])

[1, 4]


**Độ phức tạp.** `O(n)` thời gian, `O(n)` bộ nhớ. Không cần sắp xếp!

### 7.4. Cách 5 — Ngăn xếp đơn điệu (một lượt)

**Giải thích.** Mỗi mảnh được đại diện bởi **max** của nó. Duyệt `x`:

- Nếu ngăn xếp rỗng hoặc `x >= st[-1]`: `x` tự thành mảnh mới → `push(x)`.
- Ngược lại: `x` nhỏ hơn max của mảnh cuối, nên phải nhập vào mảnh đó; hơn nữa mọi mảnh trước có max `> x` cũng phải nhập vào (vì `x` phải đứng trước chúng sau khi sắp). Lưu `top = st.pop()`; pop tiếp khi `st[-1] > x`; rồi `push(top)` (max của mảnh gộp).

Đáp án = kích thước ngăn xếp.

In [14]:
def by_stack(arr):
    st = []
    for x in arr:
        if not st or x >= st[-1]:
            st.append(x)
        else:
            top = st.pop()
            while st and st[-1] > x:
                st.pop()
            st.append(top)
    return len(st)

for arr, exp in EXAMPLES:
    print(by_stack(arr), exp)

1 1
4 4


**Code nộp LeetCode (Cách 5).**

In [15]:
class SolutionStack:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        st = []                          # max của từng mảnh, không giảm
        for x in arr:
            top = x
            while st and st[-1] > x:     # x phải gộp vào các mảnh có max > x
                top = max(top, st.pop())
            st.append(top)
        return len(st)

print([SolutionStack().maxChunksToSorted(a[:]) for a, _ in EXAMPLES])

[1, 4]


Đây là bản viết gọn: `top` bắt đầu bằng `x`; nếu có pop thì `top` là max lớn nhất bị pop (chính là đỉnh đầu tiên, vì ngăn xếp không giảm).

**Độ phức tạp.** Mỗi phần tử push/pop tối đa một lần → `O(n)`. Bộ nhớ `O(n)`.

### 7.5. (Bonus) Đưa về bài I bằng hạng (rank)

Gắn `rank` (mục 3.4) biến mảng thành hoán vị `0..n−1` giữ nguyên thứ tự tương đối (trùng giá trị thì theo chỉ số). Khi đó áp dụng công thức bài 769: cắt sau `i` ⇔ `max(rank[0..i]) == i`.

Vì sao gán theo chỉ số là đúng? Hai phần tử bằng nhau không bao giờ cần "đổi chỗ", nên ép phần tử đứng trước có hạng nhỏ hơn không làm mất vết cắt nào.

In [16]:
def by_rank(arr):
    order = sorted(range(len(arr)), key=lambda i: (arr[i], i))
    rank = [0] * len(arr)
    for r, i in enumerate(order):
        rank[i] = r
    chunks = mx = 0
    for i, r in enumerate(rank):
        mx = max(mx, r)
        chunks += mx == i
    return chunks

print([by_rank(a) for a, _ in EXAMPLES])

[1, 4]


## 8. TRACE từng bước

Trace **Cách 5 (ngăn xếp)** trên `arr = [2, 1, 3, 4, 4]` và trên `arr = [1, 1, 0, 0, 1]`.

Với `[2, 1, 3, 4, 4]`:

| x | trước | hành động | sau |
|---|---|---|---|
| 2 | `[]` | push | `[2]` |
| 1 | `[2]` | 1 < 2: pop 2 (top=2), push 2 | `[2]` |
| 3 | `[2]` | push | `[2, 3]` |
| 4 | `[2, 3]` | push | `[2, 3, 4]` |
| 4 | `[2, 3, 4]` | 4 >= 4: push | `[2, 3, 4, 4]` |

Kích thước 4. ✔

In [17]:
def trace_stack(arr):
    st = []
    print(f"{'x':>3} | {'trước':<16} | hành động                      | sau")
    for x in arr:
        before = st[:]
        top = x
        popped = []
        while st and st[-1] > x:
            v = st.pop(); popped.append(v); top = max(top, v)
        st.append(top)
        act = f"pop {popped}, push max={top}" if popped else f"push {x}"
        print(f"{x:>3} | {str(before):<16} | {act:<30} | {st}")
    print("số mảnh =", len(st))
    return len(st)

trace_stack([2, 1, 3, 4, 4])
print()
trace_stack([1, 1, 0, 0, 1])

  x | trước            | hành động                      | sau
  2 | []               | push 2                         | [2]
  1 | [2]              | pop [2], push max=2            | [2]
  3 | [2]              | push 3                         | [2, 3]
  4 | [2, 3]           | push 4                         | [2, 3, 4]
  4 | [2, 3, 4]        | push 4                         | [2, 3, 4, 4]
số mảnh = 4

  x | trước            | hành động                      | sau
  1 | []               | push 1                         | [1]
  1 | [1]              | push 1                         | [1, 1]
  0 | [1, 1]           | pop [1, 1], push max=1         | [1]
  0 | [1]              | pop [1], push max=1            | [1]
  1 | [1]              | push 1                         | [1, 1]
số mảnh = 2


2

Trace **Cách 4** với bảng `cur_max` và `right_min`:

In [18]:
def trace_max_min(arr):
    n = len(arr)
    rm = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        rm[i] = min(arr[i], rm[i + 1])
    cur, chunks = float('-inf'), 0
    for i in range(n):
        cur = max(cur, arr[i])
        ok = cur <= rm[i + 1]
        chunks += ok
        print(f"i={i} arr={arr[i]:>2} cur_max={cur:>2} right_min[i+1]={rm[i + 1]!s:>4} -> {'CẮT' if ok else '   '} chunks={chunks}")
    return chunks

trace_max_min([4, 2, 2, 1, 1, 5, 6, 5])

i=0 arr= 4 cur_max= 4 right_min[i+1]=   1 ->     chunks=0
i=1 arr= 2 cur_max= 4 right_min[i+1]=   1 ->     chunks=0
i=2 arr= 2 cur_max= 4 right_min[i+1]=   1 ->     chunks=0
i=3 arr= 1 cur_max= 4 right_min[i+1]=   1 ->     chunks=0
i=4 arr= 1 cur_max= 4 right_min[i+1]=   5 -> CẮT chunks=1
i=5 arr= 5 cur_max= 5 right_min[i+1]=   5 -> CẮT chunks=2
i=6 arr= 6 cur_max= 6 right_min[i+1]=   5 ->     chunks=2
i=7 arr= 5 cur_max= 6 right_min[i+1]= inf -> CẮT chunks=3


3

## 9. Code nộp LeetCode

Bản chọn: **Cách 5 (ngăn xếp đơn điệu)** — `O(n)`, một lượt.

In [19]:
class Solution:
    def maxChunksToSorted(self, arr: List[int]) -> int:
        st = []                              # st[k] = max của mảnh thứ k; st không giảm
        for x in arr:
            top = x
            while st and st[-1] > x:         # các mảnh có max > x phải nhập với x
                top = max(top, st.pop())
            st.append(top)                   # max của mảnh (mới hoặc vừa gộp)
        return len(st)

for arr, exp in EXAMPLES:
    assert Solution().maxChunksToSorted(arr[:]) == exp
print("OK")

OK


Giải thích từng dòng:

- `st` lưu max các mảnh; vì mỗi mảnh sau phải ≥ mảnh trước (sau khi sắp), dãy max không giảm.
- `while st and st[-1] > x`: dùng **`>`** chứ không `>=` — giá trị bằng nhau không bắt buộc gộp (ví dụ `[4, 4]` là 2 mảnh).
- `top = max(top, st.pop())`: mảnh gộp có max là max lớn nhất trong các mảnh bị pop.
- `len(st)`: mỗi phần tử ngăn xếp là một mảnh.

## 10. Vì sao thuật toán đúng

**Bổ đề (đặc trưng vết cắt).** Cắt sau `i` hợp lệ ⇔ `max(arr[0..i]) <= min(arr[i+1..])`.

- (⇐) Mọi phần tử bên trái ≤ mọi phần tử bên phải, nên sau khi sắp, bên trái chính là `i+1` giá trị nhỏ nhất theo đúng thứ tự.
- (⇒) Nếu có `arr[p] > arr[q]` với `p <= i < q`, sau khi sắp `arr[p]` vẫn ở bên trái `arr[q]` → không tăng dần.

Một cách chia hợp lệ ⇔ **mọi** vết cắt của nó hợp lệ (áp dụng bổ đề cho từng vết cắt; ngược lại, nếu mọi vết cắt hợp lệ, các mảnh nằm đúng "dải giá trị" của mình). Vậy số mảnh tối đa = số vị trí hợp lệ + 1 — Cách 2, 3, 4 đếm đúng các vị trí này.

**Ngăn xếp (Cách 5) — bất biến:** sau khi xử lý tiền tố `arr[0..i]`, ngăn xếp là dãy max của các mảnh trong cách chia **tối đa** của tiền tố (xét như một mảng độc lập), không giảm. Khi thêm `x`:

- `x >= st[-1]`: mọi phần tử trước ≤ `x`, cắt trước `x` hợp lệ → thêm mảnh.
- `x < st[-1]`: mọi mảnh có max `> x` chứa phần tử lớn hơn `x` nằm bên trái `x` → không được có vết cắt giữa chúng và `x` → gộp. Các mảnh có max `<= x` vẫn tách được. Bất biến được giữ.

Cuối cùng, cách chia tối đa của tiền tố toàn mảng chính là đáp án. ∎

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n log n)` | `O(n)` |
| 2. Counter hiệu | `O(n log n)` | `O(n)` |
| 3. Tổng tiền tố | `O(n log n)` | `O(n)` |
| 4. Max/min | `O(n)` | `O(n)` |
| 5. Ngăn xếp | `O(n)` | `O(n)` |
| Bonus. Rank | `O(n log n)` | `O(n)` |

In [20]:
import random, time
random.seed(768)
tests = [[random.randint(0, 50) for _ in range(2000)] for _ in range(200)]
for name, f in [("counter", by_counter), ("prefix", by_prefix_sum), ("maxmin", by_max_min),
                ("stack", by_stack), ("rank", by_rank)]:
    t0 = time.perf_counter()
    for a in tests:
        f(a)
    print(f"{name:8} {1000 * (time.perf_counter() - t0):7.1f} ms cho 200 mảng n=2000")

counter    164.7 ms cho 200 mảng n=2000
prefix      61.4 ms cho 200 mảng n=2000
maxmin      54.6 ms cho 200 mảng n=2000
stack       25.4 ms cho 200 mảng n=2000


rank       153.6 ms cho 200 mảng n=2000


## 12. Bộ test

In [21]:
methods = {
    "counter": by_counter,
    "prefix": by_prefix_sum,
    "maxmin": by_max_min,
    "stack": by_stack,
    "rank": by_rank,
    "SolutionCounter": lambda a: SolutionCounter().maxChunksToSorted(a[:]),
    "SolutionPrefixSum": lambda a: SolutionPrefixSum().maxChunksToSorted(a[:]),
    "SolutionMaxMin": lambda a: SolutionMaxMin().maxChunksToSorted(a[:]),
    "SolutionStack": lambda a: SolutionStack().maxChunksToSorted(a[:]),
    "Solution": lambda a: Solution().maxChunksToSorted(a[:]),
}

# 12.1 Ví dụ đề
for name, f in methods.items():
    for arr, exp in EXAMPLES:
        assert f(arr) == exp, (name, arr)

# 12.2 Edge cases
edge = [
    ([7], 1),
    ([1, 1, 1, 1], 4),              # toàn bằng nhau: mỗi phần tử một mảnh
    ([1, 2, 3, 4], 4),              # đã sắp
    ([4, 3, 2, 1], 1),
    ([1, 1, 0, 0, 1], 2),
    ([0, 3, 0, 3, 2], 2),
    ([2, 1, 2, 1], 1),
    ([10 ** 8, 0], 1),
]
for name, f in methods.items():
    for arr, exp in edge:
        assert f(arr) == exp, (name, arr, f(arr))
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [22]:
# 12.3 Đối chiếu brute force (n nhỏ, nhiều giá trị trùng)
random.seed(1)
for _ in range(600):
    n = random.randint(1, 9)
    arr = [random.randint(0, 4) for _ in range(n)]
    b = brute(arr)
    for name, f in methods.items():
        assert f(arr) == b, (name, arr, f(arr), b)

# 12.4 Ngẫu nhiên lớn: các cách nhanh khớp nhau
for _ in range(300):
    n = random.randint(1, 2000)
    hi = random.choice([3, 100, 10 ** 8])
    arr = [random.randint(0, hi) for _ in range(n)]
    assert len({f(arr) for f in methods.values()}) == 1
print("Đối chiếu ngẫu nhiên: OK")

Đối chiếu ngẫu nhiên: OK


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng công thức bài I (`max == i`) trực tiếp

**Code sai:**

In [23]:
def wrong_v1_formula(arr):
    chunks = mx = 0
    for i, x in enumerate(arr):
        mx = max(mx, x)
        chunks += mx == i            # SAI: chỉ đúng khi arr là hoán vị 0..n-1
    return chunks

for a in ([2, 1, 3, 4, 4], [5, 4, 3, 2, 1]):
    print(a, "sai:", wrong_v1_formula(a), " đúng:", by_stack(a))

[2, 1, 3, 4, 4] sai: 1  đúng: 4
[5, 4, 3, 2, 1] sai: 0  đúng: 1


**Vì sao sai:** giá trị không còn là chỉ số; có trùng lặp và giá trị lớn. **Code đúng:** max/min, ngăn xếp, hoặc đổi sang rank trước (Bonus 7.5).

### Lỗi 2 — Dùng `<` thay vì `<=` ở điều kiện max/min (hoặc `>=` khi pop)

**Code sai:**

In [24]:
def wrong_strict(arr):
    n = len(arr)
    rm = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        rm[i] = min(arr[i], rm[i + 1])
    cur, chunks = float('-inf'), 0
    for i in range(n):
        cur = max(cur, arr[i])
        if cur < rm[i + 1]:          # SAI: giá trị bằng nhau vẫn cắt được
            chunks += 1
    return chunks

def wrong_stack_ge(arr):
    st = []
    for x in arr:
        top = x
        while st and st[-1] >= x:    # SAI: gộp cả mảnh có max bằng x
            top = max(top, st.pop())
        st.append(top)
    return len(st)

a = [1, 1, 1, 1]
print("strict:", wrong_strict(a), "| stack >=:", wrong_stack_ge(a), "| đúng:", by_stack(a))

strict: 1 | stack >=: 1 | đúng: 4


**Vì sao sai:** `[1, 1, 1, 1]` có thể chia 4 mảnh; điều kiện chặt làm gộp các giá trị bằng nhau một cách không cần thiết.

**Code đúng:** `cur_max <= right_min[i+1]` và `while st[-1] > x`.

### Lỗi 3 — Khi gộp, push lại `x` thay vì max của mảnh

**Code sai:**

In [25]:
def wrong_push_x(arr):
    st = []
    for x in arr:
        if not st or x >= st[-1]:
            st.append(x)
        else:
            while st and st[-1] > x:
                st.pop()
            st.append(x)             # SAI: mảnh gộp có max là đỉnh cũ, không phải x
    return len(st)

a = [3, 1, 2]
print(a, "sai:", wrong_push_x(a), " đúng:", by_stack(a))

[3, 1, 2] sai: 2  đúng: 1


**Vì sao sai:** `[3, 1, 2]`: sau khi gộp `3` với `1`, mảnh có max `3`. Nếu push `1`, phần tử `2` tưởng được mở mảnh mới (vì `2 >= 1`) — nhưng `2 < 3` nên không được.

**Code đúng:** lưu `top = st.pop()` trước khi pop tiếp, rồi push `top`.

### Lỗi 4 — Đếm "cặp ngược" / so sánh với phần tử kế bên

In [26]:
def wrong_adjacent(arr):
    return 1 + sum(1 for i in range(len(arr) - 1) if arr[i] <= arr[i + 1])   # SAI

a = [2, 1, 3, 0, 4]
print(a, "sai:", wrong_adjacent(a), " đúng:", by_stack(a))

[2, 1, 3, 0, 4] sai: 3  đúng: 2


**Vì sao sai:** điều kiện cắt là **toàn cục** (max bên trái ≤ min bên phải), không phải chỉ so hai phần tử kề nhau. `0` ở vị trí 3 kéo mọi thứ trước nó vào một mảnh.

## 14. Mẹo & nhận dạng pattern

- **"Chia mảng thành nhiều đoạn nhất sao cho … sắp riêng rồi ghép"** → vết cắt độc lập ⇒ đếm vị trí thoả `prefix_max <= suffix_min`.
- **Gộp đoạn khi phần tử mới nhỏ hơn** → ngăn xếp đơn điệu lưu **đại diện** (max) mỗi nhóm.
- **So sánh với mảng đã sắp**: đa tập tiền tố bằng nhau ⇔ tổng tiền tố bằng nhau (khi so với mảng **đã sắp**).
- **Bài có trùng lặp** → gắn chỉ số `(value, index)` để thành hoán vị.
- Họ bài: *769. Max Chunks To Make Sorted*, *915. Partition Array into Disjoint Intervals* (chính là "vết cắt đầu tiên"), *581. Shortest Unsorted Continuous Subarray*.

## 15. Biến thể

1. **Bài I (769)**: hoán vị `0..n-1` → `max(arr[0..i]) == i`.
2. **915. Partition Array into Disjoint Intervals**: trả về độ dài mảnh **đầu tiên** nhỏ nhất → vết cắt hợp lệ đầu tiên.
3. **Liệt kê các mảnh** cụ thể → ghi lại vị trí cắt.
4. **Sắp giảm dần** → đảo dấu các phần tử.
5. **Ít mảnh nhất mà mỗi mảnh dài ≤ K** → DP trên các vị trí cắt hợp lệ.

In [27]:
# Biến thể 2 (915)
def partition_disjoint(arr):
    n = len(arr)
    rm = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        rm[i] = min(arr[i], rm[i + 1])
    cur = float('-inf')
    for i in range(n - 1):
        cur = max(cur, arr[i])
        if cur <= rm[i + 1]:
            return i + 1

print(partition_disjoint([5, 0, 3, 8, 6]), partition_disjoint([1, 1, 1, 0, 6, 12]))

# Biến thể 3: liệt kê các mảnh
def list_chunks(arr):
    n = len(arr)
    rm = [float('inf')] * (n + 1)
    for i in range(n - 1, -1, -1):
        rm[i] = min(arr[i], rm[i + 1])
    res, start, cur = [], 0, float('-inf')
    for i in range(n):
        cur = max(cur, arr[i])
        if cur <= rm[i + 1]:
            res.append(arr[start:i + 1]); start = i + 1
    return res

print(list_chunks([2, 1, 3, 4, 4]), list_chunks([4, 2, 2, 1, 1, 5, 6, 5]))

3 4
[[2, 1], [3], [4], [4]] [[4, 2, 2, 1, 1], [5], [6, 5]]


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi tập vết cắt | `O(2^n n log n)` | `O(n)` | Kiểm thử |
| 2. Counter hiệu | Đa tập tiền tố == tiền tố đã sắp | `O(n log n)` | `O(n)` | Dễ nghĩ ra nhất |
| 3. Tổng tiền tố | Tổng bằng nhau ⇔ đa tập bằng nhau | `O(n log n)` | `O(n)` | Code ngắn nhất |
| 4. Max/min | `prefix_max <= suffix_min` | `O(n)` | `O(n)` | Rõ ràng, tuyến tính |
| 5. Ngăn xếp | Gộp mảnh khi gặp phần tử nhỏ hơn max | `O(n)` | `O(n)` | Một lượt, xử lý stream |

## 17. Bài tập

**Bài 1 (đoán output).** `maxChunksToSorted([1, 0, 1, 3, 2])` = ?

**Bài 2 (tính tay).** Với `arr = [4, 2, 2, 1, 1, 5, 6, 5]`, viết trạng thái ngăn xếp sau mỗi phần tử.

**Bài 3 (điền TODO).** Hoàn thiện `chunks_v1(arr)` cho bài 769 (hoán vị `0..n-1`).

**Bài 4 (tự viết lại).** Viết lại Cách 4 dùng `itertools.accumulate` cho cả tiền tố max và hậu tố min, không dùng vòng `for` tường minh để tính chúng.

**Bài 5 (mini challenge).** Viết `max_chunks_desc(arr)`: số mảnh lớn nhất sao cho sắp **giảm dần** từng mảnh rồi ghép được mảng **giảm dần**. Đối chiếu brute force.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def chunks_v1(arr):
    # TODO: đếm i sao cho max(arr[0..i]) == i
    pass

# Bài 4 — TODO
def by_max_min_accumulate(arr):
    # TODO: lm = list(accumulate(arr, max)); rm = list(accumulate(reversed(arr), min))[::-1]
    pass

# Bài 5 — TODO
def max_chunks_desc(arr):
    # TODO: gợi ý: đảo dấu
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `left_max` = 1,1,1,3,3.
- Mức 2: `right_min[i+1]` = 0,1,2,2,∞.
- Mức 3: cắt tại i=1, i=2, i=4 → `3` mảnh: `[1,0]`, `[1]`, `[3,2]`.

**Bài 2.**
- Mức 1: `4` → `[4]`; `2` < 4 → gộp → `[4]`.
- Mức 2: các số `2, 1, 1` đều gộp vào `[4]`; `5` → `[4,5]`; `6` → `[4,5,6]`.
- Mức 3: `5` < 6 → pop 6, 5 không > 5 → push 6 → `[4,5,6]`, đáp án 3.

**Bài 3.**
- Mức 1: giữ `mx`.
- Mức 2: `chunks += mx == i`.
- Mức 3: đối chiếu với `by_stack` trên hoán vị.

**Bài 4.**
- Mức 1: `accumulate(arr, max)`.
- Mức 2: hậu tố min: đảo, accumulate min, đảo lại.
- Mức 3: đếm `i` với `lm[i] <= rm[i+1]` (thêm `+∞` cuối).

**Bài 5.**
- Mức 1: giảm dần của `arr` ⇔ tăng dần của `-arr`.
- Mức 2: `by_stack([-x for x in arr])`.
- Mức 3: brute force: thay `sorted` bằng `sorted(..., reverse=True)`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().maxChunksToSorted([1, 0, 1, 3, 2]) == 3 == brute([1, 0, 1, 3, 2])

# Bài 2
def stack_states(arr):
    st, states = [], []
    for x in arr:
        top = x
        while st and st[-1] > x:
            top = max(top, st.pop())
        st.append(top)
        states.append(st[:])
    return states
assert stack_states([4, 2, 2, 1, 1, 5, 6, 5])[-1] == [4, 5, 6]

# Bài 3
def chunks_v1_ans(arr):
    chunks = mx = 0
    for i, x in enumerate(arr):
        mx = max(mx, x)
        chunks += mx == i
    return chunks

from itertools import permutations
for p in permutations(range(6)):
    assert chunks_v1_ans(list(p)) == by_stack(list(p))

# Bài 4
def by_max_min_accumulate_ans(arr):
    lm = list(accumulate(arr, max))
    rm = list(accumulate(reversed(arr), min))[::-1] + [float('inf')]
    return sum(lm[i] <= rm[i + 1] for i in range(len(arr)))

# Bài 5
def max_chunks_desc_ans(arr):
    return by_stack([-x for x in arr])

def brute_desc(arr):
    n, target, best = len(arr), sorted(arr, reverse=True), 1
    for mask in range(1 << (n - 1)):
        cuts = [i for i in range(n - 1) if mask >> i & 1]
        res, start = [], 0
        for c in cuts + [n - 1]:
            res += sorted(arr[start:c + 1], reverse=True); start = c + 1
        if res == target:
            best = max(best, len(cuts) + 1)
    return best

random.seed(9)
for _ in range(400):
    arr = [random.randint(0, 5) for _ in range(random.randint(1, 9))]
    assert by_max_min_accumulate_ans(arr) == brute(arr)
    assert max_chunks_desc_ans(arr) == brute_desc(arr), arr
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template ngăn xếp:**

```python
st = []
for x in arr:
    top = x
    while st and st[-1] > x:      # dùng '>' (bằng nhau không gộp)
        top = max(top, st.pop())
    st.append(top)
return len(st)
```

**Template max/min:** `right_min[n] = inf`; đếm `i` với `max(arr[0..i]) <= right_min[i+1]`.

**Template tổng tiền tố:** đếm `i` với `sum(arr[:i+1]) == sum(sorted(arr)[:i+1])`.

**Checklist trước khi submit:**

- [ ] Giá trị **trùng nhau** vẫn được tách (`<=` / `>`).
- [ ] Không dùng công thức bài I (`max == i`).
- [ ] Ngăn xếp: push lại **max** của mảnh gộp.
- [ ] Mảnh cuối được đếm (`+∞` hoặc `+1`).
- [ ] Đã thử: một phần tử, toàn bằng nhau, đã sắp, sắp ngược.
- [ ] Tên method đúng: `maxChunksToSorted(self, arr)`.